In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
base_folder = r"/storage/alplakes_test/geneva_100m_2025"
ke_folder = os.path.join(base_folder, "outputs_swirl", "ke_eddy")

# Prepare data

### Import datasets

In [ ]:
eddy_ke = pd.read_csv(os.path.join(ke_folder, "ke_eddies.csv"), index_col=1).drop(columns=["Unnamed: 0"])
lake_ke = pd.read_csv(os.path.join(ke_folder, "ke_lake.csv"), index_col=1).drop(columns=["Unnamed: 0"])

In [ ]:
lake_ke.index = pd.to_datetime(lake_ke.index)
eddy_ke.index = pd.to_datetime(eddy_ke.index)

In [ ]:
eddy_volume = pd.read_csv(os.path.join(base_folder, 'outputs_swirl', 'eddy_statistics', "eddy_volume_timeserie.csv"), index_col=0) * 1e6 # km3 to m3

In [ ]:
lake_mask = np.load(os.path.join(base_folder, "grid", "mask_lake.npy"))
depths = pd.read_csv(os.path.join(base_folder, "grid", "depths.csv"))

### Get lake volume without eddies

In [ ]:
grid_resolution = 100
volume_upper_lake = 0
for i in range(len(depths)):
    lake_cells = np.count_nonzero(lake_mask[i])
    volume_upper_lake += lake_cells * (grid_resolution**2) * np.abs(depths['thickness_[m]'][i])

In [ ]:
volume_without_eddies = (volume_upper_lake - eddy_volume).values.flatten()

### Get lake KE without eddies

In [ ]:
lake_ke_without_eddies = (lake_ke.values - eddy_ke.values).flatten()

# Energy density

In [ ]:
rho_w = 1000 # kg/m3

In [ ]:
ke_density_eddy_Jperkg = (1e6 * eddy_ke.values / (rho_w * eddy_volume.values)).flatten() # J/MJ x MJ / (m3 x kg/m3)

In [ ]:
ke_density_lake_Jperkg = 1e6 * lake_ke_without_eddies / (rho_w * volume_without_eddies) # J/MJ x MJ / (m3 x kg/m3)

In [ ]:
df_density_eddy = pd.DataFrame({'ke_density_Jperkg' : ke_density_eddy_Jperkg})
df_density_eddy['group']='Eddies'

In [ ]:
df_density_lake = pd.DataFrame({'ke_density_Jperkg' : ke_density_lake_Jperkg})
df_density_lake['group']='Lake without eddies'

In [ ]:
df_density_violin = pd.concat([df_density_eddy, df_density_lake])

In [ ]:
# Violin plot
plt.figure(figsize=(6, 8))
sns.violinplot(data=df_density_violin,
               y='ke_density_Jperkg',
               hue='group',
               inner='quartile',
               split=True)  # inner='quartile' shows median & quartiles
plt.legend(title=None)
plt.title('Violin Plot of Kinetic Energy density [J/kg]')
plt.ylabel('Kinetic Energy density [J/kg]')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# Energy loss & gain

In [ ]:
dKE_dt_eddies = np.roll(np.diff(eddy_ke['kinetic_energy_eddy_[MJ]'].values.flatten()), 12) #MJ/h
dKE_dt_lake = np.roll(np.diff(lake_ke_without_eddies), 12) #MJ/h

energy_diff_in_Wperkg_eddies = (1e6 * dKE_dt_eddies / 3600) / (rho_w * eddy_volume.values.flatten()[:-1]) # MJ/h / (kg/m3 * m3) = W/kg
energy_diff_in_Wperkg_lake =  (1e6 * dKE_dt_lake / 3600) / (rho_w * volume_without_eddies.flatten()[:-1]) # MJ/h / (kg/m3 * m3) = W/kg

In [ ]:
plt.plot(energy_diff_in_Wperkg_eddies[12:], label='Eddies')
plt.plot(energy_diff_in_Wperkg_lake[12:], label='Lake without eddies')
plt.legend()

In [ ]:
df_loss_eddy = pd.DataFrame({'ke_loss_Wperkg' : np.log10(-1*energy_diff_in_Wperkg_eddies[energy_diff_in_Wperkg_eddies<0])})
df_loss_eddy['group']='Eddies'

df_loss_lake = pd.DataFrame({'ke_loss_Wperkg' : np.log10(-1*energy_diff_in_Wperkg_lake[energy_diff_in_Wperkg_lake<0])})
df_loss_lake['group']='Lake without eddies'

df_loss_violin = pd.concat([df_loss_eddy, df_loss_lake])

In [ ]:
# Violin plot
plt.figure(figsize=(6, 8))
sns.violinplot(data=df_loss_violin,
               y='ke_loss_Wperkg',
               hue='group',
               inner='quartile',
               split=True)  # inner='quartile' shows median & quartiles
plt.legend(title=None)
plt.title('Violin Plot of Kinetic Energy loss [W/kg] (log10)')
plt.ylabel('log10(Kinetic Energy loss) [W/kg]')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.ylim(-13,-6)
plt.show()

In [ ]:
df_gain_eddy = pd.DataFrame({'ke_gain_Wperkg' : np.log10(energy_diff_in_Wperkg_eddies[energy_diff_in_Wperkg_eddies>0])})
df_gain_eddy['group']='Eddies'

df_gain_lake = pd.DataFrame({'ke_gain_Wperkg' : np.log10(energy_diff_in_Wperkg_lake[energy_diff_in_Wperkg_lake>0])})
df_gain_lake['group']='Lake without eddies'

df_gain_violin = pd.concat([df_gain_eddy, df_gain_lake])

In [ ]:
# Violin plot
plt.figure(figsize=(6, 8))
sns.violinplot(data=df_gain_violin,
               y='ke_gain_Wperkg',
               hue='group',
               inner='quartile',
               split=True)  # inner='quartile' shows median & quartiles
plt.legend(title=None)
plt.title('Violin Plot of Kinetic Energy gain rate [W/kg] (log10)')
plt.ylabel('log10(Rate of kinetic Energy gain) [W/kg]')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.ylim(-13,-6)
plt.show()